# Importación de librerías + para qué son

In [4]:
# Manipulación
import pandas as pd
import numpy as np

# Geoespacial
import geopandas as gpd
from shapely import wkt

# Estadística y modelos
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy import stats

# Encuestas complejas (estimación con diseño muestral)
from samplics.estimation import TaylorEstimator
from samplics.categorical import Tabulation

# Análisis factorial y confiabilidad
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
import pingouin as pg

# Utilidades
from unidecode import unidecode
import unicodedata
import warnings
from pathlib import Path

# Visualización de control (no es la entrega final; la entrega es Tableau)
import matplotlib.pyplot as plt
import seaborn as sns

C:\Users\Ana_L\AppData\Local\Temp\ipykernel_16220\3188069297.py:16: FutureWarning: samplics is archived and no longer maintained. Migrate to 'svy' (pip install svy) for the same functionality and more. Documentation: https://svylab.com/docs
  from samplics.estimation import TaylorEstimator


# Fijar semilla

In [5]:
np.random.seed(2026)

# Importación de los archivos csv normalizados

In [24]:
path=Path(r'..\outputs\encuesta_percepcion_legible.csv')
df=pd.read_csv(path, delimiter=',', encoding='utf-8', na_values=[''], keep_default_na=True)
print(df.columns.values)

['Año y mes según el periodo de recolección de cada una de las encuestas.'
 'Sector UPL correspondiente a donde fue realizada la encuesta.'
 'Código de la localidad correspondiente a donde fue realizada la encuesta.'
 'Nombre de la localidad correspondiente a donde fue realizada la encuesta.'
 'Unidad_de_Planeamiento_Local(UPL)'
 '3. ¿Cuántas personas conforman su hogar?'
 '4. ¿Cuántas personas de su hogar tienen menos de 18 años?'
 '5. ¿Cuántas  personas tienen 18 años cumplidos o más?'
 '2. Parentesco con el jefe del hogar' 'Edad de la persona.'
 'C. ¿Cuál es el nivel educativo más alto que usted ha alcanzado?'
 'D. Sexo al nacer' 'E. ¿Usted se reconoce como?' '¿Otra cuál?'
 'G. ¿Cuál es su estado civil?'
 'H. Según el recibo o servicio de energía eléctrica  ¿cuál es el estrato de esta vivienda?'
 '303. ¿Usted se considera pobre?' 'Sexo del jefe de hogar'
 'a. Limpieza general de la casa'
 'b. Mantenimiento y reparaciones del hogar' 'c. Lavado de ropa'
 'd. Planchar' 'e. Cocinar' 'f.

# Verificaciones de los archivos

## De la encuesta de percepción

In [23]:
#ver que tenga solo 19 valores en codigo_localidad
print(df['Código de la localidad correspondiente a donde fue realizada la encuesta.'].unique())

[19  5  7  4  8 11 10  3  9  6 18 15 13 17 14  2  1 16 12]


In [8]:
#ver que codigo_UPL tenga 30 valores
print(df['Unidad_de_Planeamiento_Local(UPL)'].unique())

['Arborizadora' 'Lucero' 'Usme - Entrenubes' 'Edén' 'Suba' 'Tibabuyes'
 'Engativá' 'Centro Histórico' 'Kennedy' 'Britalia' 'Fontibón' 'Tintal'
 'Patio Bonito' 'Porvenir' 'Bosa' 'Tunjuelito' 'Rafael Uribe'
 'San Cristóbal' 'Restrepo' 'Teusaquillo' 'Niza' 'Chapinero' 'Usaquén'
 'Toberín' 'Rincón de Suba' 'Tabora' 'Salitre' 'Puente Aranda'
 'Barrios Unidos' 'Torca']


In [25]:
TIPOS = {
    'k': ('K', 'Acoso sexual (Silbidos, comentarios sexuales, etc.)'),
    'l': ('L', 'Presenció casos de violencia intrafamiliar'),
    'm': ('M', 'Presenció casos de violencia contra la mujer'),
    'n': ('N', 'Presenció casos de violencia contra niños, niñas y adolescentes (NNA)'),
}
LUGAR = {
    1: 'En su residencia u otra residencia',
    2: 'En la cuadra, conjunto, barrio',
    3: 'En otro espacio público',
    4: 'En el transporte público',
    5: 'En el lugar de trabajo',
    6: 'No afrontó la situación',
}

renombres = {}
for letra, (pref, enunciado) in TIPOS.items():
    for i, lugar in LUGAR.items():
        renombres[f'{letra}. {enunciado} ({lugar})'] = f'{pref}x404_{i}'

faltan = [c for c in renombres if c not in df.columns]
assert not faltan, f'No se encontraron estas columnas:\n' + '\n'.join(faltan)

df = df.rename(columns=renombres)

#verificar empíricamente la semántica del bloque 404
BLOQUES = {
    "K": ("Acoso sexual",                 10593, 2489),
    "L": ("Violencia intrafamiliar",      10914, 2168),
    "M": ("Violencia contra la mujer",    10625, 2457),
    "N": ("Violencia contra NNA",         11370, 1712),
}
LUGARES = [1, 2, 3, 4, 5]


def a_binaria(serie):
    """Normaliza a 0/1 sin importar si viene numérica o como texto 'Si'/'No'."""
    if serie.dtype == object:
        s = (serie.astype(str)
                  .str.strip()
                  .str.lower()
                  .str.normalize("NFKD")
                  .str.encode("ascii", "ignore")
                  .str.decode("utf-8"))
        return s.map({"si": 1, "1": 1, "1.0": 1, "no": 0, "0": 0, "0.0": 0})
    return pd.to_numeric(serie, errors="coerce")


resultados = []

for pref, (nombre, esp_si, esp_no) in BLOQUES.items():
    col6 = f"{pref}x404_6"
    cols_lugar = [f"{pref}x404_{i}" for i in LUGARES]

    faltantes = [c for c in [col6] + cols_lugar if c not in df.columns]
    if faltantes:
        print(f"[{pref}] ✗ columnas ausentes: {faltantes}")
        continue

    no_afronto = a_binaria(df[col6])
    lugares = df[cols_lugar].apply(a_binaria)
    algun_lugar = (lugares.fillna(0).sum(axis=1) > 0).astype(int)
    n_lugares = lugares.fillna(0).sum(axis=1)

    n_si = int((no_afronto == 1).sum())
    n_no = int((no_afronto == 0).sum())
    n_nulos = int(no_afronto.isna().sum())

    # --- Chequeo 1: frecuencias contra el diccionario
    c1 = (n_si == esp_si) and (n_no == esp_no)

    # --- Chequeo 2: exclusividad  (_6 = 1  ⇒  ningún lugar marcado)
    violan_excl = int(((no_afronto == 1) & (algun_lugar == 1)).sum())
    c2 = violan_excl == 0

    # --- Chequeo 3: cobertura  (_6 = 0  ⇒  al menos un lugar marcado)
    huecos = int(((no_afronto == 0) & (algun_lugar == 0)).sum())
    c3 = huecos == 0

    # --- Chequeo 4: respuesta múltiple
    total_marcas = int(n_lugares.sum())
    marcas_por_persona = total_marcas / n_no if n_no else np.nan

    print(f"\n─── Bloque {pref} · {nombre} ───")
    print(f"  _6 = 1 (no afrontó) : {n_si:>6,}  ({n_si/len(df):.1%})   esperado {esp_si:,}")
    print(f"  _6 = 0 (afrontó)    : {n_no:>6,}  ({n_no/len(df):.1%})   esperado {esp_no:,}")
    print(f"  nulos en _6         : {n_nulos:>6,}")
    print(f"  {'✓' if c1 else '✗'} C1 frecuencias coinciden con el diccionario")
    print(f"  {'✓' if c2 else '✗'} C2 exclusividad          → {violan_excl:,} registros con _6=1 y algún lugar marcado")
    print(f"  {'✓' if c3 else '✗'} C3 cobertura             → {huecos:,} registros con _6=0 y ningún lugar marcado")
    print(f"  · marcas totales en _1.._5: {total_marcas:,}  ({marcas_por_persona:.2f} por persona que afrontó)")

    resultados.append({
        "bloque": pref, "nombre": nombre,
        "n_no_afronto": n_si, "n_afronto": n_no, "n_nulos": n_nulos,
        "violan_exclusividad": violan_excl, "huecos": huecos,
        "marcas_totales": total_marcas, "marcas_por_persona": round(marcas_por_persona, 2),
        "TAC_sin_ponderar": round(n_no / len(df), 4),
        "aprueba": bool(c1 and c2 and c3),
    })

# =============================================================
# Veredicto
# =============================================================
v01 = pd.DataFrame(resultados)
print("\n" + "=" * 62)
if v01["aprueba"].all():
    print("V-01 APROBADA 🟢  D-02 se confirma: '_6 = 1' es excluyente.")
    print("Solo '_6 = 0' es interpretable → la TAC se construye como está definida.")
else:
    print("V-01 NO APROBADA 🔴  Revisar D-02 ANTES de construir la TAC.")
    print("  · Si falla C2: '_6' no es excluyente y '_1.._5' significan 'dónde presenció',")
    print("    no 'dónde afrontó'. La definición de afrontamiento cambia por completo.")
    print("  · Si falla C3: hay registros sin información en el bloque; decidir si se")
    print("    excluyen del denominador de la TAC y documentarlo como enmienda.")
print("=" * 62)

display(v01)
#v01.to_csv('outputs/v01_verificacion_bloque404.csv', index=False, encoding='utf-8-sig')


─── Bloque K · Acoso sexual ───
  _6 = 1 (no afrontó) : 10,593  (81.0%)   esperado 10,593
  _6 = 0 (afrontó)    :  2,489  (19.0%)   esperado 2,489
  nulos en _6         :      0
  ✓ C1 frecuencias coinciden con el diccionario
  ✗ C2 exclusividad          → 19 registros con _6=1 y algún lugar marcado
  ✓ C3 cobertura             → 0 registros con _6=0 y ningún lugar marcado
  · marcas totales en _1.._5: 3,590  (1.44 por persona que afrontó)

─── Bloque L · Violencia intrafamiliar ───
  _6 = 1 (no afrontó) : 10,914  (83.4%)   esperado 10,914
  _6 = 0 (afrontó)    :  2,168  (16.6%)   esperado 2,168
  nulos en _6         :      0
  ✓ C1 frecuencias coinciden con el diccionario
  ✗ C2 exclusividad          → 15 registros con _6=1 y algún lugar marcado
  ✓ C3 cobertura             → 0 registros con _6=0 y ningún lugar marcado
  · marcas totales en _1.._5: 2,871  (1.32 por persona que afrontó)

─── Bloque M · Violencia contra la mujer ───
  _6 = 1 (no afrontó) : 10,625  (81.2%)   esperado 10

,bloque,nombre,n_no_afronto,n_afronto,n_nulos,violan_exclusividad,huecos,marcas_totales,marcas_por_persona,TAC_sin_ponderar,aprueba
0,K,Acoso sexual,10593,2489,0,19,0,3590,1.44,0.1903,False
1,L,Violencia intrafamiliar,10914,2168,0,15,0,2871,1.32,0.1657,False
2,M,Violencia contra la mujer,10625,2457,0,17,0,3217,1.31,0.1878,False
3,N,Violencia contra NNA,11370,1712,0,16,0,2249,1.31,0.1309,False


In [27]:
v01.to_csv('../outputs/v01_verificacion_bloque404.csv', index=False, encoding='utf-8-sig')